In [1]:
from scraper import fetch_page
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
openai = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

MODEL = "llama3.1"

In [3]:
def find_relevant_links(question, max_links=5):
    
    page = fetch_page("https://ttuhscep.edu/")

    if page is None:
        return []

    question_words = set(
        word.lower()
        for word in question.split()
        if len(word) > 2
    )

    scored_links = []

    for link in page["links"]:

        link_lower = link.lower()

        score = 0

        for word in question_words:
            if word in link_lower:
                score += 1

        if score > 0:
            scored_links.append(
                (score, link)
            )

    scored_links.sort(
        reverse=True
    )

    return [
        link
        for score, link in scored_links[:max_links]
    ]

In [5]:
def crawl_for_question(question, max_pages=5):

    links = find_relevant_links(
        question,
        max_links=max_pages
    )

    pages = []

    for link in links:

        print("Fetching:", link)

        page = fetch_page(link)

        if page is not None:
            pages.append(page)

    return pages

In [ ]:
def answer_question(question):

    pages = targeted_crawl(
        question,
        start_url="https://ttuhscep.edu/",
        max_pages=10
    )

    if not pages:
        return "I couldn't find relevant information on the TTUHSC website."

    website_information = ""

    for page in pages:

        website_information += f"""
TITLE: {page["title"]}

URL: {page["url"]}

CONTENT:
{page["text"][:8000]}

--------------------------------
"""

    messages = [
        {
            "role": "system",
            "content": """
You are a helpful assistant for Texas Tech University
Health Sciences Center El Paso.

Answer the user's question using ONLY the provided
TTUHSC website content.

IMPORTANT:
- Give the actual answer, not just links.
- Do not invent information.
- If the information cannot be found, say so.
- Include the source URL when possible.

Respond in markdown.
"""
        },
        {
            "role": "user",
            "content": f"""
Question:
{question}

Relevant TTUHSC website content:

{website_information}
"""
        }
    ]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    return response.choices[0].message.content

In [7]:
answer = answer_question(
    "What are the admission requirements for nursing?"
)

display(Markdown(answer))

Fetching: https://ttuhscep.edu/patient-care/index.aspx
Fetching: https://ttuhscep.edu/info-for/visitors/index.aspx
Fetching: https://ttuhscep.edu/info-for/faculty-and-staff.aspx
Fetching: https://ttuhscep.edu/breast-care-center/index.aspx
Fetching: https://ttuhscep.edu/academics/admissions-and-aid.aspx


I couldn't find the specific admission requirements for nursing on the provided TTUHSC El Paso webpage. However, there is a link to the "Admissions and Aid" page under the "Academics" section, which may provide information on admission requirements or general information that could be relevant to nursing programs.